# YARA Extended Threat Hunting
This notebook offers an example of performing YARA threat hunting extended with a series of actions from other ReversingLabs APIs.

**NOTE:** If pasted into a Python file in the displayed order, all code cells can also work as a Python script.

### Covered API classes
This notebook covers examples for the following API classes:
- **YARAHunting** (*TCA-0303*)
- **YARARetroHunting** (*TCA-0319*)
- **AdvancedSearch** (*TCA-0320*)
- **RHA1FunctionalSimilarity** (*TCA-0301*)
- **DynamicAnalysis** (*TCA-0207*)
- **ReanalyzeFile** (*TCA-0205*)

### Credentials
Credentials are loaded from a local file instead of being written here in plain text.
To learn how to creat the credentials file, see the **Storing and using the credentials** section in the [README file](./README.md)

### 1. Importing the required classes
First, we will import the required API classes.

In [ ]:
from ReversingLabs.SDK.ticloud import YARAHunting, AdvancedSearch, RHA1FunctionalSimilarity, DynamicAnalysis, \
	ReanalyzeFile, YARARetroHunting
from ReversingLabs.SDK.helper import NotFoundError, NoFileTypeError
from datetime import datetime, timedelta
import json

### 2. Loading the credentials
Next, we will load our Spectra Intelligence credentials from the local `credentials.json` file.
**NOTE: Instead of doing this step, you can paste your credentials while creating the Python object in the following step.**

In [ ]:
CREDENTIALS = json.load(open('credentials.json'))
USERNAME = CREDENTIALS.get("ticloud").get("username")
PASSWORD = CREDENTIALS.get("ticloud").get("password")
USER_AGENT = json.load(open('../user_agent.json'))["user_agent"]
HOST = "https://data.reversinglabs.com"

config = {
	"host": HOST,
	"username": USERNAME,
	"password": PASSWORD,
	"user_agent": USER_AGENT
}

### 3. Creating required objects
We will now crate the objects we require from the imported classes in advance.

In [ ]:
yara = YARAHunting(
	**config
)

retro = YARARetroHunting(
	**config
)

search = AdvancedSearch(
	**config
)

similarity = RHA1FunctionalSimilarity(
	**config
)

dynamic = DynamicAnalysis(
	**config
)

reanalyze = ReanalyzeFile(
	**config
)

### 4. Defining the ruleset
We must define and name our ruleset.
In this example, we will be using a ruleset that looks for the NSIS installer.

In [ ]:
RULESET_NAME = "yara_ruleset_NSIS_Installer"
RULESET_CONTENT = f"""
import "pe"

rule {RULESET_NAME}
{{
	/* a */
    meta:
        offset = "0x4031d1"
        examplar = "4313d352e0dafd1f22b6517126a655cae3b444fa758d2845eddfbe72f24f7bdd"
    strings:
        $op = {{
            81[2-3]efbeadde [2-6]
            81[2-3]496e7374 [2-6]
            81[2-3]736f6674 [2-6]
            81[2-3]4e756c6c
        }}
        $nsis = "\\xef\\xbe\\xad\\xdeNullsoftInst"
    condition:
        pe.sections[pe.section_index(@op)].characteristics & (pe.SECTION_MEM_READ | pe.SECTION_MEM_EXECUTE) and
        $nsis in (pe.overlay.offset..pe.overlay.offset+pe.overlay.size)
}}
"""

# This code ensures there was no ruleset with the same name already present in the cloud.
try:
	yara.delete_ruleset(RULESET_NAME)
except NotFoundError:
	print("Ruleset was not already present")

### 5. Crate ruleset

In [ ]:
yara.create_ruleset(RULESET_NAME, RULESET_CONTENT)

### 6. Create variables

In [ ]:
results = {}
reanalyze_hashes = []

five_days = datetime.now() - timedelta(hours=24)
timestamp = str(int(five_days.timestamp()))

### 7. Start YARA Retro Hunt
Now we will request starting the YARA Retro Hunt for our new ruleset.
Retro Hunt takes some time to complete so run the start action only once and then continue to check if Retro Hunt has finished.

In [ ]:
start = retro.start_retro_hunt(ruleset_name=RULESET_NAME)
print(start.text)

This will take some time so check the status with the following code:

In [ ]:
status = retro.check_status(ruleset_name=RULESET_NAME)
print(status.text)

If Retro Hunt for our ruleset has finished, proceed on.

### 8. Run Retro Matches Feed
Now we can run the matches feed and append the results to our `results` object.

In [ ]:
retro_feed = retro.yara_retro_matches_feed(
	time_format="timestamp",
	time_value=timestamp
)

matches_entries = retro_feed.json().get("rl").get("feed").get("entries")
for entry in matches_entries:
	key = entry.get("sha1")
	results[key] = {}
	reanalyze_hashes.append(key)

We now have a dict with hashes from the feed as keys inside. If you need to reset the value of the `results` object, run the "Create variables" block.

### 9. Advanced Search
We can run Advanced Search to find any hashes that have "*NSIS*" as their sample type. All found hashes will be added to `results` too.

In [ ]:
search_response = search.search(
	query_string="sampletype:*NSIS* classification:malicious"
)

search_entries = search_response.json().get("rl").get("web_search_api").get("entries")
for entry in search_entries:
	key = entry.get("sha1")
	results[key] = {}
	reanalyze_hashes.append(key)

### 10. Add similar hashes
We can search for functionally similar hashes for each hash in our `results` and add them to the corresponding hash.

In [ ]:
for key in results:
	try:
		similarity_response = similarity.get_similar_hashes_aggregated(hash_input=key, max_results=20, extended_results=False)
		results[key]["similar_hashes"] = similarity_response
	except (NotFoundError, NoFileTypeError):
		results[key]["similar_hashes"] = []
		continue

This action can take some time so be patient. The more hashes we have in our `results` the more time this will take.

### 11. Detonate hashes in a Dynamic Analysis sandbox
Next, we will detonate every hash from `results` on a Windows 10 sandbox.

In [ ]:
for key in results:
	dynamic.detonate_sample(sample_hash=key, platform="windows10")

### 12. Reanalyze all files
As a finishing touch, we will request reanalyzing all hashes from `results`.

In [ ]:
for key in results:
	resp = reanalyze.reanalyze_samples(sample_hashes=reanalyze_hashes)

### What did we gain here?
- We found hashes using our YARA ruleset.
- We found more hashes using Advanced Search and appended them to our results.
- We found functionally similar hashes for each hash in our results and added them to it as a list.
- We detonated all our hashes in a sandbox and requested Spectra Intelligence reanalysis for them.

Now each NSIS-related sample has been enriched and detonated.